In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/

/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능


In [ ]:
!pip install transformers
!pip install torchinfo
!pip install pytorch-lightning
!pip install torchmetrics
!pip install sentencepiece
!pip install -i https://test.pypi.org/simple/ bitsandbytes
!pip install peft
!pip install -q -U datasets scipy ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.9/776.9 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.1/806.1 kB 17.8 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement sentenecepiece (from versions: none)
ERROR: No matching distribution found for sentenecepiece
Looking in indexes: https://test.pypi.org/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 MB 19.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.

In [ ]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM, MistralForSequenceClassification

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from transformers import BitsAndBytesConfig

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=10
    batch_size=8
    weight_decay=1e-6
    max_len = 100

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'

df_train = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_train.csv', encoding = 'utf-8')
df_dev = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_dev.csv', encoding = 'utf-8')
df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_test.csv', encoding = 'utf-8')

df_train.dropna(inplace = True)
df_dev.dropna(inplace = True)
df_test.dropna(inplace = True)

df_train = df_train.sample(n=1000, random_state=42)
df_dev = df_dev.sample(n=100, random_state=42)
df_train.reset_index(drop = True, inplace = True)
df_dev.reset_index(drop = True, inplace = True)

cuda


In [ ]:
# df_0 = df_train[df_train['output'] == 0].sample(n=800, random_state=1)
# df_1 = df_train[df_train['output'] == 1].sample(n=800, random_state=1)

# df_sampled = pd.concat([df_0, df_1])

In [ ]:
compute_dtype = getattr(torch, "float16")

base_model_id = "maywell/Synatra-7B-Instruct-v0.2"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype
)

base_model = MistralForSequenceClassification.from_pretrained(base_model_id, quantization_config=bnb_config)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at maywell/Synatra-7B-Instruct-v0.2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    return_tensors='pt',
    model_max_length=args.max_len,
    add_eos_token=True
)
tokenizer.pad_token = tokenizer.eos_token

def generate_and_tokenize_prompt(data):
    prompt_text = f"""[INST] Check whether there are hate speeches in a following sentence or not. If there are no hate speeches, Answer will be 0, if there are, Answer 1. [/INST] {data["input"]}"""
    inputs = tokenizer(prompt_text,
                       return_tensors='pt',
                       truncation=True,
                       max_length=args.max_len,
                       padding='max_length',
                       add_special_tokens=True)

    input_ids = inputs['input_ids'][0]
    attention_mask = inputs['attention_mask'][0]
    label = data['output']

    return {"input_ids": input_ids, "attention_mask": attention_mask, "label": label}

class CustomDataset(Dataset):
    def __init__(self, dataset, input, output, tokenizer, max_len):
        self.dataset = dataset
        self.input = input
        self.output = output
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        data = {
            "input": self.dataset.iloc[idx][self.input],
            "output": self.dataset.iloc[idx][self.output]
        }
        result = generate_and_tokenize_prompt(data)
        return result["input_ids"], result["attention_mask"], result["label"]

    def __len__(self):
        return len(self.dataset)

train_dataset = CustomDataset(df_train, 'input', 'output', tokenizer, args.max_len)
train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
valid_dataset = CustomDataset(df_dev, 'input', 'output', tokenizer, args.max_len)
valid_loader = DataLoader(valid_dataset, batch_size=args.batch_size, shuffle=False)

In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=16, lora_dropout=0.05, bias="none"
)

model = prepare_model_for_kbit_training(base_model)
peft_model = get_peft_model(model, peft_config)

In [ ]:
class CustomPeftModelForSequenceClassification(nn.Module):
    def __init__(self, peft_model):
        super().__init__()
        self.peft_model = peft_model
        self.classifier = nn.Linear(4096, 2)  # 4096은 모델의 임베딩 크기

    def forward(self, input_ids, attention_mask):
        outputs = self.peft_model(input_ids=input_ids, attention_mask=attention_mask)
        # 첫 번째 토큰(예: [CLS] 토큰)의 로짓 사용
        first_token_logits = outputs.logits[:, 0, :]
        logits = self.classifier(first_token_logits)
        return logits

In [ ]:
# gpu 7.9 ~ 24.5 ~ 7.9 ~ 32.7 ~ 7.9

import transformers
from datetime import datetime
import torch.optim as optim
import torch.nn.functional as F

peft_model.to(device)
tokenizer.pad_token = tokenizer.eos_token

# 마지막 레이어를 2개의 출력을 가지는 선형 레이어로 교체
# custom_peft_model.lm_head = nn.Linear(4096, 2)  # in_features는 4096, out_features는 2
peft_model.config.num_labels = 2  # 클래스 수를 2로 설정
peft_model.config.pad_token_id = tokenizer.pad_token_id

loss_fn = nn.CrossEntropyLoss()
optimizer = AdamW(peft_model.parameters(), lr=args.start_lr)
scheduler = get_cosine_schedule_with_warmup(optimizer,
                                            num_warmup_steps = int(len(train_loader)*args.epochs * 0.1),
                                            num_training_steps = len(train_loader)*args.epochs)
# micro f1으로 바꾸기
def train(peft_model, data_loader, loss_fn, optimizer, scheduler):
    train_loss = 0
    target_lst = []
    pred_lst = []
    i = 0
    peft_model.train()
    pbar = tqdm(data_loader)
    for ids, mask, target in pbar:
        i += 1
        ids = ids.to(device)
        mask = mask.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        output = peft_model(ids, mask)
        print(output.logits.shape)
        loss = loss_fn(output.logits, target).to(device)
        # print(output.logits.shape) #[8,2] = [batch_size, num_label]
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step()

        target_lst.extend(target.detach().cpu().numpy())
        pred_lst.extend(output.logits.argmax(dim=1).tolist())

        # print("Predictions:", pred_lst)
        # print("Targets:", target_lst)

        pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(train_loss / i, 4)))
    train_loss = train_loss / len(data_loader)
    train_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
    train_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

    torch.cuda.empty_cache()
    gc.collect()

    return peft_model, train_loss, train_score, train_acc

def valid(peft_model, data_loader, loss_fn):
    valid_loss = 0
    target_lst = []
    pred_lst = []
    i = 0
    peft_model.eval()
    pbar = tqdm(data_loader)
    for ids, mask, target in pbar:
        i += 1
        ids = ids.to(device)
        mask = mask.to(device)
        target = target.to(device)

        output = peft_model(ids, mask)
        loss = loss_fn(output.logits, target).to(device)
        valid_loss += loss.item()

        target_lst.extend(target.detach().cpu().numpy())
        pred_lst.extend(output.logits.argmax(dim=1).tolist())

        # print("Predictions:", pred_lst)
        # print("Targets:", target_lst)

        pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(valid_loss / i, 4)))
    valid_loss = valid_loss / len(data_loader)
    valid_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
    valid_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

    torch.cuda.empty_cache()
    gc.collect()

    return peft_model, valid_loss, valid_score, valid_acc

stop_val_f1 = 0
stop_count = 0

# for epoch in range(args.epochs):
for epoch in range(args.epochs):
    if stop_count == 3:
        break

    peft_model, train_loss, train_score, train_acc = train(peft_model, train_loader, loss_fn, optimizer, scheduler)
    peft_model, valid_loss, valid_score, valid_acc = valid(peft_model, valid_loader, loss_fn)

    print(f'Epoch : {epoch + 1},    t_loss : {round(train_loss, 4)},   t_f1 : {round(train_score, 4)},   t_acc : {round(train_acc, 4)}\n')
    print(f'              v_loss : {round(valid_loss, 4)},   v_f1 : {round(valid_score, 4)},   v_acc : {round(valid_acc, 4)}\n')

    if valid_score > stop_val_f1:
        default_weight_path = path + 'ddd_synatra_7b_e.pt'
        torch.save(peft_model.state_dict(), default_weight_path)
        stop_val_f1 = valid_score
        stop_count = 0
    else:
        stop_count += 1

    torch.cuda.empty_cache()
    gc.collect()

  0%|          | 0/125 [00:00<?, ?it/s]

torch.Size([8, 2])


[C_loss : 8.4023]:   1%|          | 1/125 [00:00<01:02,  2.00it/s]

torch.Size([8, 2])


[C_loss : 7.9126]:   2%|▏         | 2/125 [00:00<00:55,  2.20it/s]

...

[C_loss : 1.0545]:  99%|█████████▉| 124/125 [00:52<00:00,  2.36it/s]

torch.Size([8, 2])


[C_loss : 0.7782]: 100%|██████████| 13/13 [00:01<00:00,  7.19it/s]


Epoch : 2,    t_loss : 1.0505,   t_f1 : 0.617,   t_acc : 0.631

              v_loss : 0.7782,   v_f1 : 0.6885,   v_acc : 0.69



KeyboardInterrupt: ignored

In [ ]:
torch.cuda.empty_cache()
gc.collect()

0

In [ ]:
import torch
from transformers import BitsAndBytesConfig

compute_dtype = getattr(torch, "float16")

base_model_id = "maywell/Synatra-V0.1-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype
)

base_model = AutoModelForSequenceClassification.from_pretrained(base_model_id, quantization_config=bnb_config)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at maywell/Synatra-V0.1-7B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    return_tensors='pt',
    model_max_length=args.max_len,
    add_eos_token=True
)
tokenizer.pad_token = tokenizer.eos_token

def test_generate_and_tokenize_prompt(data):
    prompt_text = f"""[INST] Check whether there are hate speeches in a following sentence or not. If there are no hate speeches, Answer will be 0, if there are, Answer 1. [/INST] {data["input"]}"""
    inputs = tokenizer(prompt_text,
                       return_tensors='pt',
                       truncation=True,
                       max_length=args.max_len,
                       padding='max_length',
                       add_special_tokens=True)

    input_ids = inputs['input_ids'][0]
    attention_mask = inputs['attention_mask'][0]

    return {"input_ids": input_ids, "attention_mask": attention_mask}

class TestDataset(Dataset):
    def __init__(self, dataset, text_column, tokenizer, max_len):
        self.dataset = dataset
        self.text_column = text_column
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        data = {
            "input": self.dataset.iloc[idx][self.text_column]
        }
        result = test_generate_and_tokenize_prompt(data)
        return result["input_ids"], result["attention_mask"]

    def __len__(self):
        return len(self.dataset)

test_dataset = TestDataset(df_test, 'input', tokenizer, args.max_len)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

tokenizer_config.json:   0%|          | 0.00/965 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training, PeftModel, PeftConfig

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=16, lora_dropout=0.05, bias="none"
)

model = prepare_model_for_kbit_training(base_model)
peft_model = get_peft_model(model, peft_config)

In [ ]:
# peft_model.config.num_labels = 2 # 마지막 아웃풋 레이어 크기 제한
peft_model.config.pad_token_id = tokenizer.pad_token_id

model_path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/ddd_synatra_7b_e.pt'
peft_model.load_state_dict(torch.load(model_path), strict=True)
peft_model.to(device)

peft_model.eval()
res = []
with torch.no_grad():
    for ids, mask in tqdm(test_loader):
        ids = ids.to(device)
        mask = mask.to(device)

        output = peft_model(ids, mask)
        res.extend(output.logits.argmax(dim=1).tolist())
        # print(output.logits.argmax(dim=1).shape)
        # print(output.logits.shape) # [batch_size, max_length, vocab_size]가 나와야 함
        # max_values, max_indices = torch.max(output.logits, dim=1)
        # print("Max values:", max_values) # 최대값
        # print("Max indices:", max_indices) # 최대값 인덱스

df_test['output'] = res

100%|██████████| 259/259 [00:34<00:00,  7.53it/s]


In [ ]:
res

In [ ]:
'''
        output = peft_model(ids, mask)[0]
        output = output.cpu().numpy()
        res.extend(output.argmax(axis = 1).tolist())

        output = peft_model(ids, mask)
        logits = output.logits
        preds = logits.argmax(dim=1).tolist()
        res.extend(preds)
'''

In [ ]:
# # 토크나이저의 패딩 토큰 확인
# print("Tokenizer padding token:", tokenizer.pad_token)

# # 토크나이저의 패딩 토큰 ID 확인
# pad_token_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)
# print("Tokenizer padding token ID:", pad_token_id)

# # 모델의 패딩 토큰 ID 비교
# print("Model padding token ID:",base_model.config.pad_token_id)
# if pad_token_id == model.config.pad_token_id:
#     print("Model is using the tokenizer's padding token.")
# else:
#     print("Model is NOT using the tokenizer's padding token.")

Tokenizer padding token: </s>
Tokenizer padding token ID: 2
Model padding token ID: 2
Model is using the tokenizer's padding token.


In [ ]:
df_test

,Unnamed: 0,id,input,output
0,0,nikluge-au-2022-test-000001,극좌는 이 비겁자층을 제대로 요리할 줄 안다...,1
1,1,nikluge-au-2022-test-000002,내가 진짜 올 해 안에 차 산다!,0
2,2,nikluge-au-2022-test-000003,"선거 때마다 불장난 하는 못된 버릇 대대손손 배워가지고 그러고 까불어대면, 너 나중...",0
3,3,nikluge-au-2022-test-000004,"난 99년도에 이미 세상은 망해서 선한자들은 이미 모두 하늘로 올라갔고, 남은 우리...",0
4,4,nikluge-au-2022-test-000005,피의자로 가는 싸가지 없는 쓰래기!,1
...,...,...,...,...
2067,2067,nikluge-au-2022-test-002068,근데 비계 올 사람만 찍어,0
2068,2068,nikluge-au-2022-test-002069,지들 입맛대로 막 가네 미친,1
2069,2069,nikluge-au-2022-test-002070,얘는 지가 이걸 쓰면서 왜 이런 생각을 못하지?,1
2070,2070,nikluge-au-2022-test-002071,알겟나요,0


In [ ]:
df_test.to_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/test_16_synatra_new_df_test.csv')

In [ ]:
def jsonlload(fname):
    with open(fname, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        j_list = [json.loads(line) for line in lines]
    return j_list

test_df = jsonlload('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/NIKL_AU_2023_COMPETITION_v1.0/nikluge-au-2022-test.jsonl')
len(test_df)

2072

In [ ]:
for i, p in enumerate(df_test['output']):
    test_df[i]['output'] = p

In [ ]:
def jsonldump(j_list, fname):
    with open(fname, "w", encoding='utf-8') as f:
        for json_data in j_list:
            f.write(json.dumps(json_data, ensure_ascii=False)+'\n')
jsonldump(test_df, '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/test_16_new_synatra_7b.jsonl')

In [ ]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=10
    batch_size=32
    weight_decay=1e-6
    max_len = 60

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'

df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_test.csv', encoding = 'utf-8')
df_test_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/테스트 데이터-스마일게이트/df_test.csv', encoding = 'utf-8')

df_test_pre.dropna(inplace = True)

cuda


In [ ]:
import torch
from transformers import BitsAndBytesConfig

compute_dtype = getattr(torch, "float16")

base_model_id = "maywell/Synatra-V0.1-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype
)

base_model = AutoModelForSequenceClassification.from_pretrained(base_model_id, quantization_config=bnb_config)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/4.54G [00:00<?, ?B/s]


===================================BUG REPORT===================================
Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes

 and submit this information together with your error trace to: https://github.com/TimDettmers/bitsandbytes/issues
CUDA_SETUP: WARNING! libcudart.so not found in any environmental path. Searching in backup paths...
CUDA SETUP: CUDA runtime path found: /usr/local/cuda/lib64/libcudart.so.11.0
CUDA SETUP: Highest compute capability among GPUs detected: 7.0
CUDA SETUP: Detected CUDA version 118
CUDA SETUP: Loading binary /usr/local/lib/python3.10/dist-packages/bitsandbytes/libbitsandbytes_cuda118_nocublaslt.so...


/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: /usr/lib64-nvidia did not contain ['libcudart.so', 'libcudart.so.11.0', 'libcudart.so.12.0'] as expected! Searching further paths...
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('/sys/fs/cgroup/memory.events /var/colab/cgroup/jupyter-children/memory.events')}
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('http'), PosixPath('8013'), PosixPath('//172.28.0.1')}
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('//colab.research.google.com/tun/m/cc483011

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at maywell/Synatra-V0.1-7B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from tokenizers.processors import TemplateProcessing

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    return_tensors='pt',
    model_max_length=args.max_len,
    add_eos_token=True
)
tokenizer.pad_token = tokenizer.eos_token

def generate_and_tokenize_prompt(data):
    prompt_text = f"""[INST] Check whether there are hate speeches in a following sentence or not. If there are no hate speeches, Answer will be 0, if there are, Answer 1. [/INST] {data["input"]}"""
    inputs = tokenizer(prompt_text,
                       return_tensors='pt',
                       truncation=True,
                       max_length=args.max_len,
                       padding='max_length',
                       add_special_tokens=True)

    input_ids = inputs['input_ids'][0]
    attention_mask = inputs['attention_mask'][0]
    label = data['output']

    return {"input_ids": input_ids, "attention_mask": attention_mask, "label": label}

class TestDataset(Dataset):
    def __init__(self, dataset, input, output, tokenizer, max_len):
        self.dataset = dataset
        self.input = input
        self.output = output
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        data = {
            "input": self.dataset.iloc[idx][self.input],
            "output": self.dataset.iloc[idx][self.output]
        }
        result = generate_and_tokenize_prompt(data)
        return result["input_ids"], result["attention_mask"], result["label"]

    def __len__(self):
        return len(self.dataset)

test_dataset = TestDataset(df_test_pre, 'input', 'output', tokenizer, args.max_len)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=True)

In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training, PeftModel, PeftConfig

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=16, lora_dropout=0.05, bias="none"
)

model = prepare_model_for_kbit_training(base_model)
peft_model = get_peft_model(model, peft_config)

In [ ]:
peft_model.config.pad_token_id = tokenizer.pad_token_id

model_path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/ddd_synatra_7b_e.pt'
peft_model.load_state_dict(torch.load(model_path), strict=True)
peft_model.to(device)

def evaluate_model(model, data_loader):
    target_lst = []
    pred_lst = []
    peft_model.eval()
    with torch.no_grad():
        for ids, mask, target in tqdm(data_loader):
            ids = ids.to(device)
            mask = mask.to(device)
            target = target.to(device)

            output = peft_model(ids, mask)
            pred_lst.extend(output.logits.argmax(dim=1).tolist())
            target_lst.extend(target.cpu().numpy())

    eval_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='macro')
    eval_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

    return eval_score, eval_acc

f1_score, accuracy = evaluate_model(peft_model, test_loader)
print(f"F1-Score: {f1_score}, Accuracy: {accuracy}")

100%|██████████| 66/66 [00:39<00:00,  1.66it/s]

F1-Score: 0.5832835583164099, Accuracy: 0.5867176301958911
